# Univariate Zeitreihenanalyse — EUR/USD

Analyse des EUR/USD-Wechselkurses (Ticker: `EURUSD=X`) 2015–2025.

Box-Jenkins-Methodik in 7 Schritten:

1. Visuelle Inspektion
2. Stationaritaetstest (ADF)
3. Differenzierung
4. ACF & PACF
5. Modellauswahl (p, d, q)
6. Residualdiagnostik
7. Prognose

---
## Schritt 1: Visuelle Inspektion

- Langfristiger Trend?
- Volatilitaetsphasen?
- Strukturbrueche (Brexit, COVID, EZB-Entscheidungen)?

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

from src.utils.data_loader import load_series

In [ ]:
eurusd = load_series("EURUSD=X")

In [ ]:
print("=== Datentypen & Struktur ===")
eurusd.info()
print("\n=== Erste Zeilen ===")
display(eurusd.head())
print("\n=== Statistische Kennzahlen ===")
display(eurusd.describe())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(eurusd.index, eurusd["Close"], color="#003399", linewidth=1.0, label="Schlusskurs")
ax.set_title("EUR/USD Wechselkurs 2015-2025", fontsize=14, pad=12)
ax.set_xlabel("Datum", fontsize=11)
ax.set_ylabel("Kurs (USD pro EUR)", fontsize=11)
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Erste Beobachtungen

_(Hier kommen Alpers Beobachtungen rein)_

---
## Schritt 2: Stationaritaetstest (ADF-Test)

- H0: Reihe hat Einheitswurzel (nicht stationaer)
- H1: Reihe ist stationaer

p-Wert < 0.05 -> stationaer

In [ ]:
from statsmodels.tsa.stattools import adfuller

close = eurusd["Close"].dropna()

result = adfuller(close)
print("=== ADF-Test auf die EUR/USD-Reihe ===")
print(f"ADF-Statistik : {result[0]:.4f}")
print(f"p-Wert        : {result[1]:.4f}")
print("Kritische Werte:")
for key, val in result[4].items():
    print(f"  {key}: {val:.4f}")

if result[1] < 0.05:
    print("\n-> Reihe ist stationaer")
else:
    print("\n-> Reihe ist NICHT stationaer -> Differenzierung erforderlich")

---
## Schritt 3: Differenzierung

Wechselkurse sind typischerweise nicht stationaer. Wir differenzieren einmal (d=1).

In [ ]:
close_diff = close.diff().dropna()

result_diff = adfuller(close_diff)
print("=== ADF-Test auf differenzierte Reihe ===")
print(f"ADF-Statistik : {result_diff[0]:.4f}")
print(f"p-Wert        : {result_diff[1]:.4f}")

if result_diff[1] < 0.05:
    print("\n-> Differenzierte Reihe ist stationaer -> d = 1")
else:
    print("\n-> Weitere Differenzierung notwendig")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(close_diff.index, close_diff.values, color="#003399", linewidth=0.8)
ax.set_title("EUR/USD — Erste Differenz")
ax.set_xlabel("Datum")
ax.set_ylabel("Delta Kurs")
ax.grid(True, linestyle="--", alpha=0.5)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

---
## Schritt 4: ACF & PACF

- **ACF** -> MA-Ordnung (q)
- **PACF** -> AR-Ordnung (p)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(close_diff, lags=30, ax=axes[0])
axes[0].set_title("ACF — differenzierter EUR/USD")

plot_pacf(close_diff, lags=30, ax=axes[1], method="ywm")
axes[1].set_title("PACF — differenzierter EUR/USD")

plt.tight_layout()
plt.show()

---
## Schritt 5: Modellauswahl (p, d, q)

Grid search ueber Kandidatenmodelle. Niedrigeres AIC = besseres Modell.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings("ignore")

best_aic = np.inf
best_order = None
best_model = None

candidates = [(p, 1, q) for p in range(0, 4) for q in range(0, 4)]

print("Modellvergleich (AIC):")
print("-" * 30)

for order in candidates:
    try:
        model = ARIMA(close, order=order).fit()
        print(f"ARIMA{order}  AIC: {model.aic:.2f}")
        if model.aic < best_aic:
            best_aic = model.aic
            best_order = order
            best_model = model
    except:
        pass

print(f"\n-> Bestes Modell: ARIMA{best_order} mit AIC = {best_aic:.2f}")

In [ ]:
print(best_model.summary())

---
## Schritt 6: Residualdiagnostik

Ein gutes Modell hinterlaesst weisse Residuen ohne Autokorrelation.

- Zeitplot der Residuen
- ACF der Residuen
- Ljung-Box-Test

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

residuals = best_model.resid

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(residuals.index, residuals.values, color="gray", linewidth=0.7)
axes[0].axhline(0, color="red", linestyle="--", linewidth=0.8)
axes[0].set_title("Residuen ueber die Zeit")
axes[0].set_xlabel("Datum")
axes[0].grid(True, linestyle="--", alpha=0.4)

plot_acf(residuals, lags=30, ax=axes[1])
axes[1].set_title("ACF der Residuen")

plt.tight_layout()
plt.show()

lb_test = acorr_ljungbox(residuals, lags=[10, 20], return_df=True)
print("=== Ljung-Box-Test ===")
print(lb_test)
print("\n(p > 0.05 -> keine signifikante Autokorrelation in den Residuen)")

---
## Schritt 7: Prognose

30-Tage-Prognose mit 95% Vorhersageintervall.

In [ ]:
forecast_steps = 30
forecast = best_model.get_forecast(steps=forecast_steps)
forecast_mean = forecast.predicted_mean
forecast_ci = forecast.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(close[-180:].index, close[-180:].values,
        color="#003399", linewidth=1.0, label="Historisch (letzte 180 Tage)")

ax.plot(forecast_mean.index, forecast_mean.values,
        color="orange", linewidth=1.5, linestyle="--", label="Prognose (30 Tage)")

ax.fill_between(forecast_ci.index,
                forecast_ci.iloc[:, 0],
                forecast_ci.iloc[:, 1],
                color="orange", alpha=0.2, label="95% Vorhersageintervall")

ax.set_title(f"EUR/USD — ARIMA{best_order} Prognose (30 Tage)", fontsize=13)
ax.set_xlabel("Datum")
ax.set_ylabel("Kurs (USD pro EUR)")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

print("\n=== Prognosewerte ===")
print(pd.DataFrame({"Prognose": forecast_mean, "CI_low": forecast_ci.iloc[:,0], "CI_high": forecast_ci.iloc[:,1]}))